# Reaction diagrams from an rxflow database

Set `DB_PATH` below to an HPC results database. Site comparisons are the focus: all pathways for a ligand share one absolute Gibbs-energy reference (the lowest completed reactant), so site 1 and site 2 are directly comparable. The overlay colors substrate sites and varies line style by binding permutation. For each site/permutation, it shows the completed pathway with the lowest reactant Gibbs energy. Then the notebook renders the reactant, TS and product structures for the overall lowest-reactant pathway. Endpoint structures are the lowest-Gibbs retained conformers; the TS structure is its completed optimized geometry. Energies come from ORCA NumFreq and are shown in kcal/mol.

In [ ]:
from pathlib import Path
import sqlite3
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import xyzrender
from IPython.display import display, Markdown, HTML

DB_PATH = Path("hpc_results/workflow.sqlite")  # change to your database
HARTREE_TO_KCAL = 627.509474

if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database not found: {DB_PATH.resolve()}")
with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as connection:
    cursor = connection.execute("""
        SELECT ligand_id, pair_index, seating, endpoint, conformer, status, retained,
               gibbs_free_energy_eh, binding_permutation, binding_mode,
               coordination_isomer, substrate_site, xyz_file, donor, hydrogen, acceptor
        FROM jobs
    """)
    columns = [column[0] for column in cursor.description]
    jobs = [dict(zip(columns, row)) for row in cursor.fetchall()]
print(f"Loaded {len(jobs)} job rows from {DB_PATH}")
counts = Counter((row["endpoint"], row["status"]) for row in jobs)
display([{ "endpoint": endpoint, "status": status, "rows": count }
         for (endpoint, status), count in sorted(counts.items())])

`substrate_site` is the one-based substrate atom index used by rxflow. A profile needs a successful TS and at least one successful, retained reactant and product conformer with Gibbs energies. It appears only after its jobs are no longer pending/running. Where a site has several resonance or reaction-pair records for the same binding permutation, the overlay uses the one with the lowest reactant Gibbs energy.

In [ ]:
# Keep successful results with Gibbs energies; failed or still-running rows are excluded.
keys = ("ligand_id", "pair_index", "seating", "binding_permutation")
groups = defaultdict(list)
for row in jobs:
    key = tuple(row[column] for column in keys)
    groups[key].append(row)

profiles = []
for key, all_rows in groups.items():
    if any(row["status"] in ("pending", "running") for row in all_rows):
        continue
    rows = [row for row in all_rows if row["status"] == "succeeded"
            and row["gibbs_free_energy_eh"] is not None]
    reactants = [r for r in rows if r["endpoint"] == "reactant" and r["retained"] == 1]
    products = [r for r in rows if r["endpoint"] == "product" and r["retained"] == 1]
    transition_states = [r for r in rows if r["endpoint"] == "TS"]
    if not (reactants and products and transition_states):
        continue
    reactant = min(reactants, key=lambda row: row["gibbs_free_energy_eh"])
    product = min(products, key=lambda row: row["gibbs_free_energy_eh"])
    ts = min(transition_states, key=lambda row: row["gibbs_free_energy_eh"])
    profiles.append(dict(zip(keys, key), reactant=reactant["gibbs_free_energy_eh"],
                         TS=ts["gibbs_free_energy_eh"], product=product["gibbs_free_energy_eh"],
                         reactant_row=reactant, product_row=product, ts_row=ts,
                         binding_mode=reactant["binding_mode"],
                         coordination_isomer=reactant["coordination_isomer"],
                         substrate_site=reactant["substrate_site"]))
print(f"Complete reaction profiles: {len(profiles)}")
print(f"Incomplete groups excluded: {len(groups) - len(profiles)}")

In [ ]:
if not profiles:
    print("No complete reactant–TS–product profiles yet.")
else:
    import matplotlib as mpl
    x = [0, 1, 2]
    labels = ["Reactant", "TS", "Product"]
    half_width = 0.24
    ligand_ids = dict.fromkeys(row["ligand_id"] for row in profiles)

    for ligand_id in ligand_ids:
        group = [row for row in profiles if row["ligand_id"] == ligand_id]
        reference = min(row["reactant"] for row in group)

        # For direct site/mode comparison, keep the lowest-reactant pathway per site and permutation.
        site_mode_best = {}
        for row in group:
            key = (row["substrate_site"], row["binding_permutation"])
            if key not in site_mode_best or row["reactant"] < site_mode_best[key]["reactant"]:
                site_mode_best[key] = row
        compared = list(site_mode_best.values())

        summary = []
        for row in sorted(compared, key=lambda item: (item["substrate_site"] or 0,
                                                       item["binding_permutation"] or 0)):
            r, ts, p = row["reactant"], row["TS"], row["product"]
            summary.append({
                "substrate_site": row["substrate_site"],
                "binding_permutation": row["binding_permutation"],
                "binding_mode": row["binding_mode"],
                "reactant_vs_lowest_site_kcal_mol": (r - reference) * HARTREE_TO_KCAL,
                "TS_vs_lowest_site_kcal_mol": (ts - reference) * HARTREE_TO_KCAL,
                "product_vs_lowest_site_kcal_mol": (p - reference) * HARTREE_TO_KCAL,
                "barrier_from_reactant_kcal_mol": (ts - r) * HARTREE_TO_KCAL,
                "reaction_energy_kcal_mol": (p - r) * HARTREE_TO_KCAL,
            })
        display(Markdown(f"### Ligand {ligand_id}: site-by-site comparison (kcal/mol)"))
        headers = ("Substrate site", "Permutation", "Mode", "Reactant vs lowest", "TS vs lowest",
                   "Product vs lowest", "Barrier from reactant", "Reaction energy")
        table = "<table><thead><tr>" + "".join(f"<th>{header}</th>" for header in headers) + "</tr></thead><tbody>"
        for item in summary:
            table += (f"<tr><td>{item['substrate_site']}</td><td>{item['binding_permutation']}</td>"
                      f"<td>{item['binding_mode']}</td>"
                      f"<td>{item['reactant_vs_lowest_site_kcal_mol']:+.2f}</td>"
                      f"<td>{item['TS_vs_lowest_site_kcal_mol']:+.2f}</td>"
                      f"<td>{item['product_vs_lowest_site_kcal_mol']:+.2f}</td>"
                      f"<td>{item['barrier_from_reactant_kcal_mol']:+.2f}</td>"
                      f"<td>{item['reaction_energy_kcal_mol']:+.2f}</td></tr>")
        display(HTML(table + "</tbody></table>"))

        sites = dict.fromkeys(row["substrate_site"] for row in compared)
        colors = {site: mpl.colormaps["tab10"](i % 10) for i, site in enumerate(sites)}
        permutations = dict.fromkeys(row["binding_permutation"] for row in compared)
        line_styles = {permutation: ("-", "--", ":", "-.")[i % 4]
                       for i, permutation in enumerate(permutations)}
        fig, ax = plt.subplots(figsize=(10, 6))
        for row in compared:
            energies = [row["reactant"], row["TS"], row["product"]]
            relative = [(energy - reference) * HARTREE_TO_KCAL for energy in energies]
            site = row["substrate_site"]
            permutation = row["binding_permutation"]
            color = colors[site]
            style = line_styles[permutation]
            label = (f"site {site}, perm {permutation} (mode {row['binding_mode']}, "
                     f"isomer {row['coordination_isomer']})")

            for state, (position, energy) in enumerate(zip(x, relative)):
                ax.hlines(energy, position - half_width, position + half_width,
                          color=color, linewidth=3.5, zorder=3)
                ax.scatter(position, energy, color="#D62728" if state == 1 else color,
                           edgecolor="white", linewidth=0.6, s=38, zorder=4,
                           marker="D" if state == 1 else "o",
                           label=label if state == 0 else None)
                if len(compared) <= 6:
                    ax.annotate(f"{energy:+.1f}", xy=(position, energy), xytext=(0, 8),
                                textcoords="offset points", ha="center", fontsize=8, color=color)
            for state in range(2):
                ax.plot([x[state] + half_width, x[state + 1] - half_width],
                        [relative[state], relative[state + 1]],
                        color=color, linestyle=style, linewidth=1.5, zorder=2)

        ax.axhline(0, color="black", linestyle=":", linewidth=1, alpha=0.6)
        ax.set_xticks(x, labels)
        ax.set_xlim(-0.55, 2.55)
        ax.set_ylabel(r"Gibbs free energy relative to lowest reactant, $\Delta G$ (kcal/mol)", fontsize=12)
        ax.set_xlabel("Reaction coordinate", fontsize=12)
        ax.set_title(f"Ligand {ligand_id}: substrate-site comparison", fontsize=14)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(axis="y", alpha=0.2)
        ax.legend(title="Substrate site and binding permutation", fontsize=8,
                  bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        plt.show()

        # Render the structures for the completed pathway with the lowest reactant Gibbs energy.
        lowest = min(group, key=lambda row: row["reactant"])
        display(Markdown(
            f"### Lowest-reactant pathway: site {lowest['substrate_site']}, "
            f"permutation {lowest['binding_permutation']}, pair {lowest['pair_index']}"
        ))
        for title, row_key in (("Reactant", "reactant_row"), ("Transition state", "ts_row"),
                               ("Product", "product_row")):
            structure = lowest[row_key]
            xyz_path = DB_PATH.resolve().parent / structure["xyz_file"]
            display(Markdown(f"**{title}** — $G = {structure['gibbs_free_energy_eh']:.8f}$ Eh"))
            if not xyz_path.is_file():
                print(f"XYZ not found: {xyz_path}")
                continue
            render_options = {"no_hy": True, "bo": False}
            if row_key == "ts_row":
                reacting_atoms = (structure["donor"], structure["hydrogen"], structure["acceptor"])
                if all(atom is not None for atom in reacting_atoms):
                    donor, hydrogen, acceptor = (atom + 1 for atom in reacting_atoms)
                    render_options["ts_bonds"] = [(donor, hydrogen), (hydrogen, acceptor)]
            display(xyzrender.render(xyz_path, **render_options))